### RAG Pipelines-Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\Productiv\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)
Ignoring wrong 

Found 2 PDF files to process

Processing: model.pdf


Ignoring wrong pointing object 99 0 (offset 0)


  ✓ Loaded 12 pages

Processing: nickel.pdf
  ✓ Loaded 8 pages

Total documents loaded: 20


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content="Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume 9, Nomor 1, Juni 2023  e-ISSN: 2598-5841   \nDOI: 10.34128/jsi.v9i1.523 Received: 11 November 2022  Accepted: 27 Juni 2023  77 \nPemodelan Sistem Deteksi Kadar Unsur Hara Tanah Berdasarkan Nilai NPK Menggunakan Metode Fuzzy Mamdani  Dityo Kreshna Argeshwara1), Zulkham Umar Rosyidin2), Aji Prasetya Wibawa3),  Anik Nur Handayani4), Mokh. Sholihul Hadi5)  12345) Universitas Negeri Malang, Jl. Semarang No. 5 Malang, Jawa Timur, Indonesia 1) dityo.kreshna354@gmail.com 2) zzhulqam@gmail.com 3)aji.prasetya.ft@um.ac.id 4)aniknur.ft@um.ac.id 5)mokh.sholihul.ft@um.ac.id  Abstrak  Profil kesuburan tanah merupa

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 20 documents into 83 chunks

Example chunk:
Content: Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume 9, Nomor 1, Juni 2023  e-ISSN: 2598-5841   
DOI: 10.34128/jsi.v9i1.523 Received: 11 November 2022  Accepted: 27 Juni 2023  77...
Metadata: {'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume 9, Nomor 1, Juni 2023  e-ISSN: 2598-5841   \nDOI: 10.34128/jsi.v9i1.523 Received: 11 November 2022  Accepted: 27 Juni 2023  77'),
 Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Pemodelan Sistem Deteksi Kadar Unsur Hara Tanah Berdasarkan Nilai NPK Menggunakan Metode Fuzzy Mamdani  Dityo Kreshna Argeshwara1), Zulkh

## embeding And VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7753.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8016\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStrore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 618


In [9]:
chunks

[Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume 9, Nomor 1, Juni 2023  e-ISSN: 2598-5841   \nDOI: 10.34128/jsi.v9i1.523 Received: 11 November 2022  Accepted: 27 Juni 2023  77'),
 Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20230811022238Z00'00'", 'moddate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Pemodelan Sistem Deteksi Kadar Unsur Hara Tanah Berdasarkan Nilai NPK Menggunakan Metode Fuzzy Mamdani  Dityo Kreshna Argeshwara1), Zulkh

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 83 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.25s/it]


Generated embeddings with shape: (83, 384)
Adding 83 documents to vector store...
Successfully added 83 documents to vector store
Total documents in collection: 701


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Jurnal Sains dan Informatika ")

Retrieving documents for query: 'Jurnal Sains dan Informatika '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': '78694940-0c42-4fa7-9efd-6175173cf5e9',
  'content': 'Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume X, Nomor X, Juni/November 2020  e-ISSN: 2598-5841  \n 86',
  'metadata': {'page_label': '10',
   'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext',
   'creator': 'PyPDF',
   'doc_index': 34,
   'source': '..\\data\\pdf\\9.+Zulkham+.pdf',
   'total_pages': 12,
   'creationdate': "D:20230811022238Z00'00'",
   'source_file': '9.+Zulkham+.pdf',
   'file_type': 'pdf',
   'content_length': 110,
   'page': 9,
   'moddate': "D:20230811022238Z00'00'"},
  'similarity_score': 0.40602177381515503,
  'distance': 0.593978226184845,
  'rank': 1},
 {'id': '1af07ca4-c6ca-4081-9e3f-e78a1bc92825',
  'content': 'Jurnal Sains dan Informatika  p-ISSN: 2460-173X Volume X, Nomor X, Juni/November 2020  e-ISSN: 2598-5841  \n 86',
  'metadata': {'total_pages': 12,
   'creationdate': "D:20230811022238Z00'00'",
   'source': '..\\data\\pdf\\9.+Zulkham+.pdf',
   'page_label': '10',
  

### Integration Vectordb context pipeline With LLM Output

In [25]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#initialize the groq LLm (Set GROQ_API_KEY in env)
groq_api_key = os.getenv('GROQ_API_KEY')

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


  

In [26]:
answer=rag_simple("Jurnal Sains dan Informatika", rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Jurnal Sains dan Informatika'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Jurnal Sains dan Informatika**  
- p‑ISSN: 2460‑173X  
- e‑ISSN: 2598‑5841  
- Volume X, Nomor X (Juni/November 2020)


### Enhance RAG pipeline Features

In [28]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Long Short Term Memory", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Long Short Term Memory'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Long Short‑Term Memory (LSTM)** is a recurrent neural network (RNN) architecture designed to overcome the vanishing‑gradient problem of vanilla RNNs. It introduces a memory cell and three gating mechanisms—input, forget, and output gates—that regulate the flow of information. This structure allows LSTMs to capture long‑range dependencies in sequential data, making them a standard choice for tasks such as language modeling, machine translation, and other sequence‑to‑sequence problems.
Sources: [{'source': 'attention.pdf', 'page': 10, 'score': 0.31941545009613037, 'preview': 'tional sequence to sequence learning. arXiv preprint arXiv:1705.03122v2, 2017.\n[10] Alex Graves. Generating sequences with recurrent neural networks. arXiv preprint\narXiv:1308.0850, 2013.\n[11] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im-\nage recognition. In P...'}, {'source': 'attention.pdf', 'page': 1, 'score': 0.27530115842819214, 'preview': '1 Introduction\nR

In [31]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is al

l you need

Answer:

Final Answer: **Attention Is All You Need** is the 2017 paper that introduced the Transformer architecture. It shows that a neural network can learn to translate (and perform other sequence tasks) using only self‑attention mechanisms—no recurrent or convolutional layers—yielding faster training and better performance.

Citations:
[1] attention.pdf (page 2)
Summary: The 2017 paper “Attention Is All You Need” introduced the Transformer architecture, which relies solely on self‑attention mechanisms rather than recurrent or convolutional layers. This design enables faster training and improved performance on translation and other sequence tasks.
History: {'question': 'what is attention is all you need', 'answer': '**Attention Is All You Need** is the 2017 paper that introduced the Transformer architecture. It shows that a neural network can learn to translate (and perform other sequence tasks) using only self‑attention mechanisms—no recurrent or convolutional layers—yi